In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Dependencies

In [ ]:
# Change to your desired folder within Google Drive
%cd /content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability

# Verify the current working directory
!pwd

# Install dependencies from requirements.txt
!pip install -r "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/requirements.txt"

/content/drive/.shortcut-targets-by-id/1bYzySW5IufDYqjNZ__j7p1XADTFW12vS/6.4610: Project/Controllable Readability
/content/drive/.shortcut-targets-by-id/1bYzySW5IufDYqjNZ__j7p1XADTFW12vS/6.4610: Project/Controllable Readability
  Using cached accelerate-0.19.0-py3-none-any.whl.metadata (16 kB)
Using cached accelerate-0.19.0-py3-none-any.whl (219 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.21.0
    Uninstalling accelerate-0.21.0:
      Successfully uninstalled accelerate-0.21.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.0 requires accelerate>=0.21.0, but you have accelerate 0.19.0 which is incompatible.


In [ ]:
# !pip install --upgrade datasets fsspec huggingface_hub

In [ ]:
pip install accelerate==0.21.0

  Using cached accelerate-0.21.0-py3-none-any.whl.metadata (17 kB)
Using cached accelerate-0.21.0-py3-none-any.whl (244 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.19.0
    Uninstalling accelerate-0.19.0:
      Successfully uninstalled accelerate-0.19.0


In [ ]:
# import nltk
# nltk.download('punkt_tab')

# Preprocessing

In [ ]:
!python src/preprocess/preprocess_cnndm.py

README.md: 15.6kB [00:00, 15.7MB/s]
3.0.0/train-00000-of-00003.parquet: 100% 257M/257M [00:02<00:00, 103MB/s]
3.0.0/train-00001-of-00003.parquet: 100% 257M/257M [00:06<00:00, 39.9MB/s]
3.0.0/train-00002-of-00003.parquet: 100% 259M/259M [00:02<00:00, 101MB/s]
3.0.0/validation-00000-of-00001.parquet: 100% 34.7M/34.7M [00:00<00:00, 60.8MB/s]
3.0.0/test-00000-of-00001.parquet: 100% 30.0M/30.0M [00:01<00:00, 19.2MB/s]
Generating train split: 100% 287113/287113 [00:11<00:00, 25491.03 examples/s]
Generating validation split: 100% 13368/13368 [00:00<00:00, 25224.55 examples/s]
Generating test split: 100% 11490/11490 [00:00<00:00, 22302.81 examples/s]
Starting processing for train split...
Finished processing and saved 287113 entries for train split to data/train.json
Starting processing for validation split...
Finished processing and saved 13368 entries for validation split to data/validation.json
Starting processing for test split...
Finished processing and saved 11490 entries for test split 

In [ ]:
# Generates prompts based on approximate grade level
import json
from tqdm import tqdm
import os

BASE_PATH = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/data"

def open_txt_file(file):
    entities = []
    for line in open(file).readlines():
        entities.append(line)
    return entities


def open_file(file):
    entities = []
    for line in open(file).readlines():
        entities.append(json.loads(line))
    return entities


def save_file(data, file):
    with open(file, 'w') as file_writer:
        for line in data:
            file_writer.write(json.dumps(line) + "\n")


def get_prompt(flesch_summary):
    if flesch_summary >= 80:
        return 'Write highlights for this article for a 11 years old student:\n\n'
    elif flesch_summary >= 60:
        return 'Write highlights for this article for a middle school student:\n\n'
    elif flesch_summary >= 40:
        return 'Write highlights for this article for a high school student:\n\n'
    else:
        return 'Write highlights for this article for a college student:\n\n'


def transform_data(split):
    input_path = os.path.join(BASE_PATH, f"{split}.json")
    output_path = os.path.join(BASE_PATH, f"{split}_prompt_category.json")

    data = open_file(input_path)
    new_data = []

    for entry in tqdm(data):
        flesch_summary = entry["summary_metrics"]["flesch"]
        prompt = get_prompt(flesch_summary)

        entry["prompt"] = prompt
        entry["input_noprompt"] = entry["input"]
        entry["input"] = prompt + entry["input"]

        new_data.append(entry)

    save_file(new_data, output_path)


transform_data('train')
transform_data('validation')
transform_data('test')


100%|██████████| 11490/11490 [00:00<00:00, 241298.17it/s]


In [ ]:
# Generates prompts based on exact Flesch score, might not be necessary
def get_prompt(flesch_summary):
    return (
        'Write highlights for this article with a flesch kincaid score of '
        + str(int(round(flesch_summary, 0)))
        + ":\n\n"
    )


def transform_data(split):
    input_path = os.path.join(BASE_PATH, f"{split}.json")
    output_path = os.path.join(BASE_PATH, f"{split}_prompt_score.json")

    data = open_file(input_path)
    new_data = []

    for entry in tqdm(data):

        flesch_summary = entry["summary_metrics"]["flesch"]
        flesch_input = entry["input_metrics"]["flesch"]

        prompt = get_prompt(flesch_summary)
        entry["prompt"] = prompt
        entry["input_noprompt"] = entry["input"]
        entry["input"] = prompt + entry["input"]

        # Only skip test entries where input FK >= 50
        if split == 'test' and flesch_input >= 50:
            continue

        new_data.append(entry)

    save_file(new_data, output_path)


transform_data('train')
transform_data('validation')
transform_data('test')


100%|██████████| 11490/11490 [00:00<00:00, 262438.07it/s]


# Training


In [ ]:
# import os

# # Path to your dataset
# BASE_DATA = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/data"
# TRAIN_FILE = os.path.join(BASE_DATA, "train_prompt_category.json")
# VAL_FILE   = os.path.join(BASE_DATA, "validation_prompt_category.json")

# MODEL_NAME = "google/flan-t5-large"

# # Save inside your project folder
# BASE_PROJECT = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability"
# OUTPUT_DIR = os.path.join(BASE_PROJECT, "checkpoints/flan_t5_category_run")
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Train
# !deepspeed \
#   run_summarization_fix.py \
#   --model_name_or_path $MODEL_NAME \
#   --output_dir $OUTPUT_DIR \
#   --text_column input \
#   --summary_column summary \
#   --train_file $TRAIN_FILE \
#   --validation_file $VAL_FILE \
#   --learning_rate 1e-4 \
#   --max_source_length 1024 \
#   --source_prefix "" \
#   --num_train_epochs 20 \
#   --logging_steps 20 \
#   --preprocessing_num_workers 2 \
#   --eval_steps 500 \
#   --save_steps 500 \
#   --save_total_limit 2 \
#   --evaluation_strategy "steps" \
#   --per_device_train_batch_size 1 \
#   --per_device_eval_batch_size 1 \
#   --metric_for_best_model "rouge1" \
#   --load_best_model_at_end \
#   --predict_with_generate \
#   --deepspeed ds_config_stage3_fb16.json \
#   --bf16 \
#   --bf16_full_eval \
#   --do_train


[2025-11-28 17:44:00,791] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
2025-11-28 17:44:04.680070: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-28 17:44:04.697886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764351844.719565   17314 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764351844.726597   17314 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764351844.

In [ ]:
import json
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AdamW
from transformers import get_scheduler
from tqdm import tqdm
from rouge_score import rouge_scorer

# -----------------------
# Config
# -----------------------
MODEL_NAME = "google/flan-t5-large"
TRAIN_FILE = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/data/train_prompt_category.json"
VALIDATION_FILE = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/data/validation_prompt_category.json"
OUTPUT_DIR = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/checkpoints/flan_t5_category_run"
BATCH_SIZE = 1
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 256
NUM_EPOCHS = 4
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_SUBSAMPLE = 10000  # Use ~1/20th of the training set
VAL_SUBSAMPLE = 1000     # Use 2k examples for validation

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------
# Dataset
# -----------------------
class JSONDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.data = []
        with open(path, 'r') as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tokenizer = tokenizer
        # Set the source and target column names explicitly
        self.source_column = "input"
        self.target_column = "summary"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        source = item[self.source_column]
        target = item[self.target_column]

        source_enc = self.tokenizer(
            source, max_length=MAX_SOURCE_LENGTH, truncation=True, padding="max_length", return_tensors="pt"
        )
        target_enc = self.tokenizer(
            target, max_length=MAX_TARGET_LENGTH, truncation=True, padding="max_length", return_tensors="pt"
        )

        return {
            "input_ids": source_enc["input_ids"].squeeze(),
            "attention_mask": source_enc["attention_mask"].squeeze(),
            "labels": target_enc["input_ids"].squeeze(),
        }

# -----------------------
# Load tokenizer & model
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

# -----------------------
# Datasets
# -----------------------
train_dataset = JSONDataset(TRAIN_FILE, tokenizer)
val_dataset = JSONDataset(VALIDATION_FILE, tokenizer)

# -----------------------
# Subsample datasets
# -----------------------
train_indices = random.sample(range(len(train_dataset)), TRAIN_SUBSAMPLE)
val_indices = random.sample(range(len(val_dataset)), VAL_SUBSAMPLE)

train_dataset = Subset(train_dataset, train_indices)
val_dataset = Subset(val_dataset, val_indices)

# -----------------------
# Dataloaders
# -----------------------
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# -----------------------
# Optimizer & Scheduler
# -----------------------
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
num_training_steps = NUM_EPOCHS * len(train_loader)
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# -----------------------
# Training loop
# -----------------------
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

best_rouge1 = 0.0
global_step = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        global_step += 1

        if global_step % 2000 == 0:
            print(f"Step {global_step} - Loss: {loss.item():.4f}")

        # Save checkpoint every 500 steps
        if global_step % 2000 == 0:
            checkpoint_path = os.path.join(OUTPUT_DIR, f"checkpoint-{global_step}")
            model.save_pretrained(checkpoint_path)
            tokenizer.save_pretrained(checkpoint_path)

    # -----------------------
    # Validation
    # -----------------------
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            generated_ids = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=MAX_TARGET_LENGTH)
            preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            refs = tokenizer.batch_decode(labels, skip_special_tokens=True)

            all_preds.extend(preds)
            all_labels.extend(refs)

    # Compute Rouge1
    rouge1_scores = [scorer.score(ref, pred)['rouge1'].fmeasure for ref, pred in zip(all_labels, all_preds)]
    avg_rouge1 = sum(rouge1_scores) / len(rouge1_scores)
    print(f"Epoch {epoch+1} - Validation Rouge1: {avg_rouge1:.4f}")

    # Save best model
    if avg_rouge1 > best_rouge1:
        best_rouge1 = avg_rouge1
        best_model_path = os.path.join(OUTPUT_DIR, "best_model")
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"Saved new best model with Rouge1: {best_rouge1:.4f}")


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Training Epoch 1:  20%|█▉        | 1999/10000 [13:00<52:46,  2.53it/s]

Step 2000 - Loss: 0.4530


Training Epoch 1:  40%|███▉      | 3999/10000 [26:13<39:07,  2.56it/s]

Step 4000 - Loss: 0.3656


Training Epoch 1:  60%|█████▉    | 5999/10000 [39:28<26:04,  2.56it/s]

...

Step 38000 - Loss: 0.2060


Training Epoch 4: 100%|█████████▉| 9999/10000 [1:05:40<00:00,  2.51it/s]

Step 40000 - Loss: 0.2641


Evaluating: 100%|██████████| 1000/1000 [54:53<00:00,  3.29s/it]


Epoch 4 - Validation Rouge1: 0.4210
Saved new best model with Rouge1: 0.4210


# Baseline Inference

In [ ]:
import json

DOC_FILE = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/data/test_prompt_category.json"

# open_file logic
documents = []
with open(DOC_FILE, "r") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            print(f"Line {i} is empty")
            continue
        try:
            doc = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Line {i} failed to parse: {e}")
            continue
        if "input_noprompt" not in doc:
            print(f"Line {i} missing 'input_noprompt' field")
            continue
        documents.append(doc["input_noprompt"])

print(f"Loaded {len(documents)} documents successfully")
# optionally print first few
for d in documents[:5]:
    print(d)


Loaded 11490 documents successfully
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from scorer import FleschScorer
from lookahead import Lookahead
from generation import Generator
from tqdm import tqdm
import json
import torch
import os

# ---------------- CONFIG ----------------
LOOKAHEAD_LENGTH = 1
DOC_FILE = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/data/test_prompt_category.json"
MODEL_PATH = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/checkpoints/flan_t5_category_run/best_model"
BATCH_SIZE = 1
MAX_INPUT_LENGTH = 1024
MAX_OUTPUT_LENGTH = 256

# ---------------- LOAD DATA ----------------
def open_file(file):
    entities = []
    with open(file, "r") as f:
        for line in f:
            entities.append(json.loads(line))
    return entities

raw_docs = open_file(DOC_FILE)
base_documents = [doc["input_noprompt"] for doc in raw_docs]
print(f"Loaded {len(base_documents)} raw documents.")

# (Optional) reduce to first 400 docs
base_documents = base_documents[:1]
print(f"Using {len(base_documents)} documents.")

# ---------------- LOAD MODEL ----------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, local_files_only=True).cuda()

# ---------------- GENERATION FUNCTION ----------------
def run_generation(prompt, score_target, output_path):
    print(f"\n===== Running generation: {output_path} =====")

    documents = [prompt + doc for doc in base_documents]

    scorer = FleschScorer('flesch', score_target)
    lookahead = Lookahead(
        model=model,
        tokenizer=tokenizer,
        scorer=scorer,
        lookahead_length=LOOKAHEAD_LENGTH,
        lookahead_lambda=25,
        lookahead_top_k=5,
        decoding_type="greedy",
        max_length=MAX_OUTPUT_LENGTH
    )
    generator = Generator(model, lookahead=lookahead)

    summaries = []
    os.makedirs("outputs", exist_ok=True)

    for i in tqdm(range(0, len(documents), BATCH_SIZE)):
        batch_docs = documents[i:i+BATCH_SIZE]

        inputs = tokenizer(batch_docs, max_length=MAX_INPUT_LENGTH,
                           padding=True, truncation=True, return_tensors="pt")
        inputs = {k: v.cuda() for k, v in inputs.items()}

        output_obj = generator.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=MAX_OUTPUT_LENGTH,
            return_dict_in_generate=True
        )

        decoded = tokenizer.batch_decode(output_obj.sequences, skip_special_tokens=True)
        summaries.extend(decoded)

    with open(f"outputs/{output_path}", "w") as f:
        for line in summaries:
            f.write(line + "\n")

    print(f"Saved → outputs/{output_path}")


# ---------------- RUN ALL 4 VERSIONS ----------------
run_generation(
    prompt="Write highlights for this article for an 11 years old student:\n\n",
    score_target=90,
    output_path="11yold_test3.txt"
)

# run_generation(
#     prompt="Write highlights for this article for a middle school student:\n\n",
#     score_target=70,
#     output_path="middle-school.txt"
# )

# run_generation(
#     prompt="Write highlights for this article for a high school student:\n\n",
#     score_target=50,
#     output_path="high-school.txt"
# )

# run_generation(
#     prompt="Write highlights for this article for a college student:\n\n",
#     score_target=30,
#     output_path="college-student.txt"
# )


Loaded 11490 raw documents.
Using 1 documents.

===== Running generation: 11yold_test3.txt =====


100%|██████████| 1/1 [00:07<00:00,  7.36s/it]

Saved → outputs/11yold_test3.txt


# Altered Inference

In [ ]:
import json

DOC_FILE = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/data/test_prompt_category.json"

# open_file logic
documents = []
with open(DOC_FILE, "r") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            print(f"Line {i} is empty")
            continue
        try:
            doc = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Line {i} failed to parse: {e}")
            continue
        if "input_noprompt" not in doc:
            print(f"Line {i} missing 'input_noprompt' field")
            continue
        documents.append(doc["input_noprompt"])

print(f"Loaded {len(documents)} documents successfully")
# optionally print first few
for d in documents[:5]:
    print(d)


Loaded 11490 documents successfully
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from scorer import FleschScorer
from lookahead import Lookahead
from generation import Generator
from tqdm import tqdm
#from textblob import TextBlob
import json
import torch
import os

# ---------------- CONFIG ----------------
LOOKAHEAD_LENGTH = 1
DOC_FILE = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/controllable-readability-summarization/data/test_prompt_category.json"
MODEL_PATH = "/content/drive/MyDrive/2025 Fall/6.4610/6.4610: Project/Controllable Readability/checkpoints/flan_t5_category_run/best-model"
BATCH_SIZE = 1
MAX_INPUT_LENGTH = 1024
MAX_OUTPUT_LENGTH = 256

# ---------------- LOAD DATA ----------------
def open_file(file):
    entities = []
    with open(file, "r") as f:
        for line in f:
            entities.append(json.loads(line))
    return entities

raw_docs = open_file(DOC_FILE)
base_documents = [doc["input_noprompt"] for doc in raw_docs]
print(f"Loaded {len(base_documents)} raw documents.")

# (Optional) reduce to first 400 docs
base_documents = base_documents[:1]
print(f"Using {len(base_documents)} documents.")

# ---------------- LOAD MODEL ----------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, local_files_only=True).cuda()

# ---------------- GENERATION FUNCTION ----------------
def run_generation(prompt, score_target, output_path):
    print(f"\n===== Running generation: {output_path} =====")

    documents = [prompt + doc for doc in base_documents]

    scorer = FleschScorer('flesch', score_target)
    lookahead = Lookahead(
        model=model,
        tokenizer=tokenizer,
        scorer=scorer,
        lookahead_length=LOOKAHEAD_LENGTH,
        lookahead_lambda=25,
        lookahead_top_k=5,
        decoding_type="greedy",
        max_length=MAX_OUTPUT_LENGTH
    )
    generator = Generator(model, lookahead=lookahead)

    summaries = []
    os.makedirs("outputs", exist_ok=True)

    for i in tqdm(range(0, len(documents), BATCH_SIZE)):
        batch_docs = documents[i:i+BATCH_SIZE]

        inputs = tokenizer(batch_docs, max_length=MAX_INPUT_LENGTH,
                           padding=True, truncation=True, return_tensors="pt")
        inputs = {k: v.cuda() for k, v in inputs.items()}

        output_obj = generator.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=MAX_OUTPUT_LENGTH,
            return_dict_in_generate=True
        )

        decoded = tokenizer.batch_decode(output_obj.sequences, skip_special_tokens=True)
        summaries.extend(decoded)

    with open(f"outputs/{output_path}", "w") as f:
        for line in summaries:
            f.write(line + "\n")

    print(f"Saved → outputs/{output_path}")


# ---------------- RUN ALL 4 VERSIONS ----------------
# run_generation(
#     prompt="Write highlights for this article for an 11 years old student:\n\n",
#     score_target=90,
#     output_path="11yold_test_example1.txt"
# )

# run_generation(
#     prompt="Write highlights for this article for a middle school student:\n\n",
#     score_target=70,
#     output_path="middle-school_bias95.txt"
# )

# run_generation(
#     prompt="Write highlights for this article for a high school student:\n\n",
#     score_target=50,
#     output_path="high-school_bias95.txt"
# )

# run_generation(
#     prompt="Write highlights for this article for a college student:\n\n",
#     score_target=30,
#     output_path="college-student_bias95.txt"
# )


Loaded 11490 raw documents.
Using 1 documents.


In [ ]:
# ---------------- QUICK SANITY CHECK ----------------
print("\nRunning single-example test...")

test_prompt = "Write highlights for this article for an 11 years old student:\n\n"
test_doc = base_documents[0]
test_input = test_prompt + test_doc

print(test_input)
scorer = FleschScorer("flesch", 90)

lookahead = Lookahead(
    model=model,
    tokenizer=tokenizer,
    scorer=scorer,
    lookahead_length=2,
    lookahead_lambda=25,
    lookahead_top_k=5,
    decoding_type="greedy",
    max_length=256
)

generator = Generator(model, lookahead=lookahead)

inputs = tokenizer([test_input], max_length=1024, padding=True, truncation=True, return_tensors="pt")
inputs = {k: v.cuda() for k, v in inputs.items()}

output_obj = generator.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=256,
    return_dict_in_generate=True
)

summary = tokenizer.decode(output_obj.sequences[0], skip_special_tokens=True)
print("\n=== SAMPLE OUTPUT ===")
print(summary)

print("\n=== FK SCORE ===")
print(scorer(summary))
print("=====================\n")



Running single-example test...
Write highlights for this article for an 11 years old student:

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to joi

TypeError: 'FleschScorer' object is not callable